# Inside the Tokenizer

In this lab, I explore how tokenizers split text into tokens before the text is sent to a large language model.  
The goal is to compare different tokenizers, investigate unusual tokenization cases, and build a small token and cost estimator.

In [1]:
# Setup
!pip install -r requirements.txt

import tiktoken
from transformers import AutoTokenizer

   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 6.3 MB/s  0:00:00



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1 — Tokenize and compare

For this task, I use one mixed-content paragraph that includes English text, a Python code line, Azerbaijani text, and emojis.  
Then I tokenize the same text with two different tokenizers and compare their tokens, token IDs, and token counts.

In [3]:
TEXT = """
I am learning how tokenizers work before sending text to an LLM.
Here is a small Python line: for i in range(3): print("hello", i)
Azerbaijani text: Salam, mən süni intellekt öyrənirəm.
Emoji test: 🚀😊
"""

print(TEXT)


I am learning how tokenizers work before sending text to an LLM.
Here is a small Python line: for i in range(3): print("hello", i)
Azerbaijani text: Salam, mən süni intellekt öyrənirəm.
Emoji test: 🚀😊



In [4]:
enc = tiktoken.get_encoding("cl100k_base")

tiktoken_ids = enc.encode(TEXT)

tiktoken_tokens = [
    enc.decode_single_token_bytes(token_id).decode("utf-8", errors="replace")
    for token_id in tiktoken_ids
]

print("=== tiktoken / cl100k_base ===")
print("Token count:", len(tiktoken_ids))
print()
print("Tokens:")
print(tiktoken_tokens)
print()
print("Token IDs:")
print(tiktoken_ids)

=== tiktoken / cl100k_base ===
Token count: 70

Tokens:
['\n', 'I', ' am', ' learning', ' how', ' token', 'izers', ' work', ' before', ' sending', ' text', ' to', ' an', ' L', 'LM', '.\n', 'Here', ' is', ' a', ' small', ' Python', ' line', ':', ' for', ' i', ' in', ' range', '(', '3', '):', ' print', '("', 'hello', '",', ' i', ')\n', 'A', 'zerbai', 'j', 'ani', ' text', ':', ' Sal', 'am', ',', ' m', 'ə', 'n', ' sü', 'ni', ' int', 'elle', 'kt', ' ö', 'yr', 'ə', 'n', 'ir', 'ə', 'm', '.\n', 'Emoji', ' test', ':', ' �', '�', '�', '�', '�', '\n']

Token IDs:
[198, 40, 1097, 6975, 1268, 4037, 12509, 990, 1603, 11889, 1495, 311, 459, 445, 11237, 627, 8586, 374, 264, 2678, 13325, 1584, 25, 369, 602, 304, 2134, 7, 18, 1680, 1194, 446, 15339, 498, 602, 340, 32, 57860, 73, 5676, 1495, 25, 8375, 309, 11, 296, 99638, 77, 60839, 7907, 528, 6853, 5964, 17372, 11160, 99638, 77, 404, 99638, 76, 627, 93831, 1296, 25, 11410, 248, 222, 76460, 232, 198]


In [5]:
hf_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

hf_ids = hf_tokenizer.encode(TEXT, add_special_tokens=False)
hf_tokens = hf_tokenizer.convert_ids_to_tokens(hf_ids)

print("=== Hugging Face / bert-base-uncased ===")
print("Token count:", len(hf_ids))
print()
print("Tokens:")
print(hf_tokens)
print()
print("Token IDs:")
print(hf_ids)

=== Hugging Face / bert-base-uncased ===
Token count: 66

Tokens:
['i', 'am', 'learning', 'how', 'token', '##izer', '##s', 'work', 'before', 'sending', 'text', 'to', 'an', 'll', '##m', '.', 'here', 'is', 'a', 'small', 'python', 'line', ':', 'for', 'i', 'in', 'range', '(', '3', ')', ':', 'print', '(', '"', 'hello', '"', ',', 'i', ')', 'azerbaijani', 'text', ':', 'sal', '##am', ',', 'm', '##ə', '##n', 'sun', '##i', 'intel', '##le', '##kt', 'o', '##yr', '##ə', '##nir', '##ə', '##m', '.', 'em', '##oj', '##i', 'test', ':', '[UNK]']

Token IDs:
[1045, 2572, 4083, 2129, 19204, 17629, 2015, 2147, 2077, 6016, 3793, 2000, 2019, 2222, 2213, 1012, 2182, 2003, 1037, 2235, 18750, 2240, 1024, 2005, 1045, 1999, 2846, 1006, 1017, 1007, 1024, 6140, 1006, 1000, 7592, 1000, 1010, 1045, 1007, 18325, 3793, 1024, 16183, 3286, 1010, 1049, 29681, 2078, 3103, 2072, 13420, 2571, 25509, 1051, 12541, 29681, 29339, 29681, 2213, 1012, 7861, 29147, 2072, 3231, 1024, 100]


In [6]:
print("| Tokenizer | Token count |")
print("|---|---:|")
print(f"| tiktoken / cl100k_base | {len(tiktoken_ids)} |")
print(f"| Hugging Face / bert-base-uncased | {len(hf_ids)} |")

| Tokenizer | Token count |
|---|---:|
| tiktoken / cl100k_base | 70 |
| Hugging Face / bert-base-uncased | 66 |


### Observation


The two tokenizers produced different counts for the same text: `tiktoken / cl100k_base` produced 70 tokens, while `bert-base-uncased` produced 66 tokens.

They disagree most on emojis, Azerbaijani words, and punctuation in the code line. For example, BERT represents the emojis as `[UNK]`, while `tiktoken` breaks them into several byte-level pieces. BERT also lowercases the text and uses WordPiece markers such as `##`, while `tiktoken` keeps leading spaces inside many tokens, such as `" learning"` or `" Python"`.

This matters because token count depends on the tokenizer, not only on the visible length of the text. The same input can use a different amount of context window and have a different API cost depending on which model tokenizer is used.

## Task 2 — Hunt the weird cases

In this task, I investigate several cases where tokenizers behave in surprising ways.  
I compare token counts for common words, rare words, English and Azerbaijani sentences, code, prose, leading spaces, emojis, and accented characters.

In [7]:
cases = {
    "common_word": "running",
    "rare_word": "antidisestablishmentarianism",
    # TODO: add at least one more pair you want to compare
}
for name, s in cases.items():
    print(f"{name:15} {len(enc.encode(s)):3}  {s!r}")

common_word       1  'running'
rare_word         6  'antidisestablishmentarianism'


In [8]:
def token_count(text):
    return len(enc.encode(text))


cases = {
    # 1. Common word vs rare / made-up word
    "common_word": "international",
    "rare_made_up_word": "internationalizationisticlymadeupword",

    # 2. English vs Azerbaijani
    "english_sentence": "I am learning machine learning at school.",
    "azerbaijani_sentence": "Mən məktəbdə maşın öyrənməsi öyrənirəm.",

    # 3. Prose vs code
    "prose": "This function adds two numbers and returns the result.",
    "code": "def add(a, b):\n    return a + b",

    # 4. Word with and without leading space
    "hello_no_space": "hello",
    "hello_with_space": " hello",

    # 5. Plain text vs emoji / accented characters
    "plain_text": "rocket smile",
    "emoji_text": "🚀😊",
    "accented_text": "süni intellekt öyrənirəm",
}

print("| Case | Token count | Text |")
print("|---|---:|---|")

for name, text in cases.items():
    print(f"| {name} | {token_count(text)} | {text!r} |")

| Case | Token count | Text |
|---|---:|---|
| common_word | 1 | 'international' |
| rare_made_up_word | 7 | 'internationalizationisticlymadeupword' |
| english_sentence | 8 | 'I am learning machine learning at school.' |
| azerbaijani_sentence | 26 | 'Mən məktəbdə maşın öyrənməsi öyrənirəm.' |
| prose | 10 | 'This function adds two numbers and returns the result.' |
| code | 11 | 'def add(a, b):\n    return a + b' |
| hello_no_space | 1 | 'hello' |
| hello_with_space | 1 | ' hello' |
| plain_text | 2 | 'rocket smile' |
| emoji_text | 5 | '🚀😊' |
| accented_text | 13 | 'süni intellekt öyrənirəm' |


Observations:

In this task, I used the `tiktoken` tokenizer with the `cl100k_base` encoding to count tokens.

1. The common word `"international"` used only 1 token, while the made-up word `"internationalizationisticlymadeupword"` used 7 tokens. This shows that rare or artificial words are split into smaller subword pieces, which increases token usage.

2. The English sentence used 8 tokens, while the Azerbaijani sentence used 26 tokens. This shows that Azerbaijani text can take more tokens because the tokenizer is more optimized for common English patterns and may split accented or less frequent words into smaller pieces.

3. The prose sentence used 10 tokens, while the code snippet used 11 tokens. Even though the code is short, symbols like parentheses, commas, newlines, indentation, and operators also affect token count.

4. `"hello"` and `" hello"` both used 1 token in this tokenizer. This shows that `cl100k_base` has separate efficient tokens for some common words with and without a leading space.

5. `"rocket smile"` used 2 tokens, while the emojis `"🚀😊"` used 5 tokens. This shows that emojis can require multiple tokens because they are represented using byte-level pieces.

6. The accented Azerbaijani text `"süni intellekt öyrənirəm"` used 13 tokens. This matters because accented and multilingual text can use more context window space than a similar-looking English phrase.

## Task 3 — Token + cost estimator

In this task, I build a small cost estimator using the `tiktoken / cl100k_base` tokenizer.  
The function counts input tokens, estimates input cost and expected output cost, then returns the projected total cost.

In [10]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    """Return (input_token_count, projected_total_cost)."""
    input_tokens = len(enc.encode(text))

    input_cost = (input_tokens / 1000) * price_in_per_1k
    output_cost = (expected_output_tokens / 1000) * price_out_per_1k
    total_cost = input_cost + output_cost

    print("Input token count:", input_tokens)
    print("Expected output tokens:", expected_output_tokens)
    print("Input cost:", round(input_cost, 6))
    print("Output cost:", round(output_cost, 6))
    print("Projected total cost:", round(total_cost, 6))

    return input_tokens, total_cost


In [11]:
price_in = 0.075
price_out = 0.30

short_question = "How do I reset my password?"

long_document = """
Tokenizers are an important part of large language model systems.
Before a model receives text, the tokenizer converts the text into smaller units called tokens.
These tokens can be whole words, parts of words, punctuation marks, spaces, emojis, or byte-level pieces.
Understanding tokenization helps developers estimate context length, API cost, and model behavior.
This is especially important when working with long documents, code files, multilingual text, or user-generated content.
"""

code_file = """
import numpy as np

def normalize(x):
    mean = np.mean(x)
    std = np.std(x)
    return (x - mean) / std

data = np.array([10, 20, 30, 40, 50])
print(normalize(data))
"""

In [12]:
print("=== Short question ===")
short_result = estimate(short_question, price_in, price_out, expected_output_tokens=50)

print("\n=== Long document ===")
long_result = estimate(long_document, price_in, price_out, expected_output_tokens=150)

print("\n=== Code file ===")
code_result = estimate(code_file, price_in, price_out, expected_output_tokens=100)

=== Short question ===
Input token count: 7
Expected output tokens: 50
Input cost: 0.000525
Output cost: 0.015
Projected total cost: 0.015525

=== Long document ===
Input token count: 91
Expected output tokens: 150
Input cost: 0.006825
Output cost: 0.045
Projected total cost: 0.051825

=== Code file ===
Input token count: 58
Expected output tokens: 100
Input cost: 0.00435
Output cost: 0.03
Projected total cost: 0.03435


In [13]:
results = [
    ("Short question", short_result[0], short_result[1]),
    ("Long document", long_result[0], long_result[1]),
    ("Code file", code_result[0], code_result[1]),
]

print("| Input type | Input tokens | Projected total cost |")
print("|---|---:|---:|")

for name, tokens, cost in results:
    print(f"| {name} | {tokens} | {cost:.6f} |")

| Input type | Input tokens | Projected total cost |
|---|---:|---:|
| Short question | 7 | 0.015525 |
| Long document | 91 | 0.051825 |
| Code file | 58 | 0.034350 |


### Summary

I used a small made-up price table: 0.075 per 1K input tokens and 0.30 per 1K output tokens.

The most expensive input was the long document. It used 91 input tokens and had a projected total cost of 0.051825.

This was expected because the long document had the highest input token count and also the largest expected output length. The code file was the second most expensive with 58 input tokens and a projected cost of 0.034350, while the short question was the cheapest with only 7 input tokens and a projected cost of 0.015525.

This shows that both input length and expected output length affect the final cost. Longer prompts and longer expected answers use more tokens, which increases API cost and context window usage.